# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedumer1941/Flyrank-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring**

This notebook generates the ranked action queue from the Random Forest model (ML-08),
assigns reason codes and action labels, documents intended use and limits, and specifies
what a human must review before acting.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# --- Setup (same as ML-08) ---
URL = "https://raw.githubusercontent.com/ahmedumer1941/Flyrank-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f"Loaded {len(df)} rows | {df['client_id'].nunique()} clients | {df['is_declining'].mean()*100:.1f}% declining")

# Feature engineering
num = ['search_volume','competition','cpc','word_count','char_count',
       'content_age_days','days_since_last_update',
       'impressions_90d','clicks_90d','pageviews_90d','sessions_90d',
       'users_90d','engaged_sessions_90d','ai_sessions_90d','scroll_events_90d',
       'days_with_impressions','days_with_sessions',
       'ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']
X_num = df[num].copy()
for c in ['search_volume','competition','cpc','word_count','char_count']:
    # Use df['content_type'] to group as it is not in X_num
    X_num[c] = X_num[c].fillna(df.groupby('content_type')[c].transform('median'))
    X_num[c] = X_num[c].fillna(X_num[c].median())

X_num.loc[X_num['avg_position']==0,'avg_position'] = X_num['avg_position'].median()
X_num['ctr_gap'] = (X_num['ctr'].max()-X_num['ctr'])/(X_num['ctr'].max()-X_num['ctr'].min()+1e-6)
X_num['staleness_w'] = X_num['days_since_last_update']/(X_num['content_age_days']+1e-6)
X_num['eng_per_session'] = X_num['engaged_sessions_90d']/(X_num['sessions_90d']+1e-6)

cat = ['content_type','main_intent','competition_level','age_tier',
       'freshness_tier','impression_tier','word_count_tier','position_tier']
X_cat = pd.get_dummies(df[cat], drop_first=True)
X = pd.concat([X_num, X_cat], axis=1).values
y = df['is_declining'].values

# Train on ALL data for final scoring
rf = RandomForestClassifier(n_estimators=100, max_depth=12,
                             random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X, y)
scores = rf.predict_proba(X)[:, 1]
print(f"Trained Random Forest on all {len(X)} rows\n")

# Build ranked queue
queue = df[['content_id','client_id','days_since_last_update','impressions_90d',
            'ctr','avg_position','trend_direction']].copy()
queue['decline_probability'] = scores
queue = queue.sort_values('decline_probability', ascending=False).reset_index(drop=True)
queue['rank'] = range(1, len(queue)+1)

# Assign action labels and reason codes
def assign_action(prob, stale, impressions):
    if prob >= 0.8:
        return 'REFRESH_CONTENT', 'HIGH_DECLINE_PROB'
    elif prob >= 0.6:
        if stale > 90:
            return 'REFRESH_STALE', 'STALE_AND_DECLINING'
        return 'OPTIMIZE_META', 'MODERATE_DECLINE_PROB'
    elif prob >= 0.4:
        if impressions > 1000:
            return 'MONITOR_HIGH_VALUE', 'HIGH_IMPRESSIONS_AT_RISK'
        return 'MONITOR', 'LOW_PRIORITY'
    else:
        return 'NO_ACTION', 'STABLE'

tmp = queue.apply(lambda r: assign_action(r['decline_probability'],
    r['days_since_last_update'], r['impressions_90d']), axis=1, result_type='expand')
queue['action_label'] = tmp[0]
queue['reason_code'] = tmp[1]

print("=== QUEUE STATISTICS ===")
print(f"Total scored pages: {len(queue)}")
for label, grp in queue.groupby('action_label'):
    print(f"  {label:25s}: {len(grp):,} pages")
print()

print("=== TOP 20 RANKED QUEUE ===")
print(f"{'Rank':>5s} | {'Decline Prob':>13s} | {'Action Label':18s} | {'Reason Code':26s} | {'Staleness':>9s} | {'Impressions':>11s}")
print("-"*95)
for _, row in queue.head(20).iterrows():
    print(f"{row['rank']:5d} | {row['decline_probability']:13.3f} | {row['action_label']:18s} | {row['reason_code']:26s} | {int(row['days_since_last_update']):>4}d  | {int(row['impressions_90d']):>8,}")

Loaded 30000 rows | 32 clients | 54.2% declining
Trained Random Forest on all 30000 rows

=== QUEUE STATISTICS ===
Total scored pages: 30000
  MONITOR                  : 4,565 pages
  MONITOR_HIGH_VALUE       : 4,614 pages
  NO_ACTION                : 8,536 pages
  OPTIMIZE_META            : 6,630 pages
  REFRESH_CONTENT          : 1,839 pages
  REFRESH_STALE            : 3,816 pages

=== TOP 20 RANKED QUEUE ===
 Rank |  Decline Prob | Action Label       | Reason Code                | Staleness | Impressions
-----------------------------------------------------------------------------------------------
    1 |         0.971 | REFRESH_CONTENT    | HIGH_DECLINE_PROB          |   20d  |   21,533
    2 |         0.968 | REFRESH_CONTENT    | HIGH_DECLINE_PROB          |   20d  |   12,892
    3 |         0.968 | REFRESH_CONTENT    | HIGH_DECLINE_PROB          |   20d  |   22,537
    4 |         0.968 | REFRESH_CONTENT    | HIGH_DECLINE_PROB          |   20d  |   17,237
    5 |         0.964 

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
docs = """
=== INTENDED USE ===

WHO:
  Content editors and SEO managers reviewing a portfolio of published pages.

WHAT FOR:
  Prioritization support: which pages to review first when planning a content
  refresh cycle. The queue reduces 30,000 pages to a manageable ordered list.

HOW:
  1. Open the top-50 queue
  2. For each page, verify the decline signal (check Google Search Console)
  3. Read the page itself \u2014 is the information outdated? Is the keyword still relevant?
  4. Apply editorial judgment: refresh, redirect, consolidate, or leave alone
  5. Log the action taken and mark the page for follow-up in 30 days

=== LIMITS ===

WHERE IT STOPS BEING VALID:
  - Does NOT replace human editorial review. The model scores probability of decline
    based on historical signals, not content quality or topical relevance.
  - Does NOT account for seasonality. A page about seasonal topics may show
    cyclical declines that are not content quality issues.
  - Trained on cross-sectional data (one 90-day snapshot). Performance drift after
    data distribution changes is expected.
  - 32 clients in training. Predictions for entirely different content verticals
    may not transfer.
  - Does NOT use keyword-level topical signals. Two pages on the same topic with the
    same metrics get the same score regardless of which one has better content.
"""
print(docs)


=== INTENDED USE ===

WHO:
  Content editors and SEO managers reviewing a portfolio of published pages.

WHAT FOR:
  Prioritization support: which pages to review first when planning a content
  refresh cycle. The queue reduces 30,000 pages to a manageable ordered list.

HOW:
  1. Open the top-50 queue
  2. For each page, verify the decline signal (check Google Search Console)
  3. Read the page itself — is the information outdated? Is the keyword still relevant?
  4. Apply editorial judgment: refresh, redirect, consolidate, or leave alone
  5. Log the action taken and mark the page for follow-up in 30 days

=== LIMITS ===

WHERE IT STOPS BEING VALID:
  - Does NOT replace human editorial review. The model scores probability of decline
    based on historical signals, not content quality or topical relevance.
  - Does NOT account for seasonality. A page about seasonal topics may show
    cyclical declines that are not content quality issues.
  - Trained on cross-sectional data (one 90-

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
checklist = """
=== HUMAN REVIEW CHECKLIST ===

Before acting on any page flagged in the top-50 queue, verify:

[ ] Open the page itself: is the content actually outdated or incorrect?
[ ] Check Google Search Console for the specific query and position history
[ ] Is the target keyword still relevant? (search volume may have shifted)
[ ] Is the content in a seasonal category? (e.g., holiday, weather, annual events)
[ ] Does the page have backlinks that make a 301 redirect better than refreshing?
[ ] Is there a newer page on the same topic that already serves the user intent?
[ ] Does the page need a full rewrite, a section update, or just a meta refresh?

=== NEVER AUTOMATE ===

The following decisions must NEVER be made without human judgment:

  1. DELETING a page \u2014 the model has no concept of backlink value or redirect cost
  2. 301 REDIRECTING without editorial review \u2014 wrong redirects destroy SEO equity
  3. NOINDEX / NOFOLLOW decisions \u2014 these are strategic, not performance-based
  4. Bulk content changes across clients \u2014 each site's content strategy differs
  5. Automatically publishing AI-generated rewrites \u2014 quality and accuracy vary
"""
print(checklist)


=== HUMAN REVIEW CHECKLIST ===

Before acting on any page flagged in the top-50 queue, verify:

[ ] Open the page itself: is the content actually outdated or incorrect?
[ ] Check Google Search Console for the specific query and position history
[ ] Is the target keyword still relevant? (search volume may have shifted)
[ ] Is the content in a seasonal category? (e.g., holiday, weather, annual events)
[ ] Does the page have backlinks that make a 301 redirect better than refreshing?
[ ] Is there a newer page on the same topic that already serves the user intent?
[ ] Does the page need a full rewrite, a section update, or just a meta refresh?

=== NEVER AUTOMATE ===

The following decisions must NEVER be made without human judgment:

  1. DELETING a page — the model has no concept of backlink value or redirect cost
  2. 301 REDIRECTING without editorial review — wrong redirects destroy SEO equity
  3. NOINDEX / NOFOLLOW decisions — these are strategic, not performance-based
  4. Bulk cont

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
monitoring = """
=== MONITORING & RETRAIN TRIGGERS ===

RETRAIN WHEN ANY OF THESE FIRE:

  1. Queue acceptance rate drops below 40% for 2 consecutive weeks
     (editors reject the model's top picks more often than they accept them)

  2. Precision@100 on fresh label data drops more than 0.10 from the 0.790 baseline
     (retrain with the latest 90-day window)

  3. New content types or client industries enter the portfolio
     (the model was trained on 32 specific clients; domain shift degrades accuracy)

  4. Google algorithm update changes how rankings/impressions behave
     (a confirmed core update is a natural retrain signal)

  5. 6 months have passed since last training
     (time-based refresh regardless of other signals)

MONITORING CADENCE:
  - Queue quality: review after each refresh cycle (weekly or biweekly)
  - Model metrics: re-evaluate Precision@K and ROC-AUC monthly
  - Feature drift: check for distribution shifts in top-5 features quarterly
"""
print(monitoring)


=== MONITORING & RETRAIN TRIGGERS ===

RETRAIN WHEN ANY OF THESE FIRE:

  1. Queue acceptance rate drops below 40% for 2 consecutive weeks
     (editors reject the model's top picks more often than they accept them)

  2. Precision@100 on fresh label data drops more than 0.10 from the 0.790 baseline
     (retrain with the latest 90-day window)

  3. New content types or client industries enter the portfolio
     (the model was trained on 32 specific clients; domain shift degrades accuracy)

  4. Google algorithm update changes how rankings/impressions behave
     (a confirmed core update is a natural retrain signal)

  5. 6 months have passed since last training
     (time-based refresh regardless of other signals)

MONITORING CADENCE:
  - Queue quality: review after each refresh cycle (weekly or biweekly)
  - Model metrics: re-evaluate Precision@K and ROC-AUC monthly
  - Feature drift: check for distribution shifts in top-5 features quarterly



## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
import os
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('../work/outputs', exist_ok=True)

# Export the full ranked queue
out_dir = 'work/outputs' if os.path.exists('work') else '../work/outputs'
queue.to_csv(f'{out_dir}/refresh_action_queue.csv', index=False)
print(f"Exported queue to: {out_dir}/refresh_action_queue.csv")

# Export a top-50 preview for the paper
queue.head(50).to_csv(f'{out_dir}/top_50_queue_preview.csv', index=False)
print(f"Exported top-50 to: {out_dir}/top_50_queue_preview.csv")

print("\u2713 Exports complete.")

Exported queue to: work/outputs/refresh_action_queue.csv
Exported top-50 to: work/outputs/top_50_queue_preview.csv
✓ Exports complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.